In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix



Ready


In [2]:
df = pd.read_csv("HR_comma_sep.csv")


In [3]:
print(df.head())
print("Shape:", df.shape)
df.info()
print("Missing values:", df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())


   satisfaction_level  last_evaluation  number_project  average_montly_hours  \
0                0.38             0.53               2                   157   
1                0.80             0.86               5                   262   
2                0.10             0.77               6                   247   
3                0.92             0.85               5                   259   
4                0.89             1.00               5                   224   

   time_spend_company  Work_accident  left  promotion_last_5years Department  \
0                   3              0     1                      0      sales   
1                   6              0     1                      0      sales   
2                   4              0     1                      0      sales   
3                   5              0     1                      0      sales   
4                   5              0     1                      0      sales   

   salary  
0     low  
1  medium  
2 

In [5]:
df = df.drop_duplicates().reset_index(drop=True)
X = df.drop(columns="left")
y = df["left"]
print("After cleaning:", df.shape)
print(y.value_counts())
print(X.dtypes)


After cleaning: (11991, 10)
left
0    10000
1     1991
Name: count, dtype: int64
satisfaction_level       float64
last_evaluation          float64
number_project             int64
average_montly_hours       int64
time_spend_company         int64
Work_accident              int64
promotion_last_5years      int64
Department                   str
salary                       str
dtype: object


## 3. Train/Test Split
`stratify=y` က class အချိုးနီးစပ်အောင် ခွဲပေးသည်။


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=25, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)
print(y_train.value_counts(normalize=True))


Train: (8393, 9) Test: (3598, 9)
left
0    0.833909
1    0.166091
Name: proportion, dtype: float64


## 4. Encoding + Scaling + Logistic Regression
One-hot encoding က စာသားအမျိုးအစားများကို numeric columns အဖြစ် ပြောင်းသည်။ Scaling က numeric features များ၏ scale ကိုညှိသည်။
Pipeline က preprocessing နဲ့ model ကို အစဉ်လိုက်ချိတ်ပေးသည်။


In [7]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()
preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), num_cols),
    ("category", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])
model = make_pipeline(preprocess, LogisticRegression(max_iter=2000, random_state=25))
model.fit(X_train, y_train)
print("Model trained")


Model trained


C:\Users\Dell\AppData\Local\Temp\ipykernel_21860\2501017548.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


## 5. Label နှင့် Probability
`predict()` က label ပေးသည်။ `predict_proba()` က class တစ်ခုချင်းစီအတွက် probability ပေးသည်။ Column အစဉ်ကို `classes_` နဲ့ စစ်ပါ။


In [8]:
print("Class order:", model.classes_)
print("Labels:", model.predict(X_test.head(5)))
print(pd.DataFrame(model.predict_proba(X_test.head(5)), columns=model.classes_))


Class order: [0 1]
Labels: [0 1 0 0 0]
          0         1
0  0.597792  0.402208
1  0.425697  0.574303
2  0.760140  0.239860
3  0.701371  0.298629
4  0.599688  0.400312


## 6. Evaluation
Confusion matrix မှာ row က actual၊ column က prediction ဖြစ်သည်။ Positive class ကို `left=1` ဟုယူပါ။
- Precision: အလုပ်ထွက်မည်ဟု ခန့်မှန်းသူများထဲ တကယ်ထွက်သူ အချိုး။
- Recall: တကယ်ထွက်သူများထဲ ဘယ်လောက်ရှာတွေ့သလဲ။
- F1: precision နဲ့ recall နှစ်ခုကို ချိန်ညှိဖော်ပြသော score။
- Accuracy: အားလုံးထဲ မှန်တဲ့အချိုး။ Class မညီမျှရင် accuracy တစ်ခုတည်းနဲ့ မဆုံးဖြတ်ပါနဲ့။


In [12]:
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))
labels = model.classes_
cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=labels),
                  index=[f"Actual {x}" for x in labels],
                  columns=[f"Predicted {x}" for x in labels])
print(cm)


Accuracy: 0.8351862145636465
              precision    recall  f1-score   support

           0       0.86      0.96      0.91      3001
           1       0.51      0.19      0.28       597

    accuracy                           0.84      3598
   macro avg       0.68      0.58      0.59      3598
weighted avg       0.80      0.84      0.80      3598

          Predicted 0  Predicted 1
Actual 0         2890          111
Actual 1          482          115


In [14]:
from IPython.display import display

demo = X_test.head(3).copy()
result = demo.copy()

result["Actual_left"] = y_test.loc[demo.index]
result["Predicted_left"] = model.predict(demo)

display(result)

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,Department,salary,Actual_left,Predicted_left
242,0.37,0.48,2,159,3,0,0,sales,low,1,0
7860,0.18,0.73,4,219,5,0,0,sales,medium,0,1
8927,0.58,0.98,3,175,3,0,0,sales,low,0,0
